<a href="https://colab.research.google.com/github/Charlestonjm21/MotionPrivacyResearch/blob/main/MotionPM_IdentityLeakage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install numpy matplotlib scikit-learn torch

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

dataset_path = "/content/drive/MyDrive/MotionPrivacyResearch/datasets/nturgbd_skeletons_s001_to_s017"

files = os.listdir(dataset_path)

print("Total files:", len(files))
print(files[:5])

Total files: 1
['nturgb+d_skeletons']


In [4]:
os.listdir("/content/drive/MyDrive/MotionPrivacyResearch")

['datasets', 'Code', 'results']

In [5]:
os.listdir("/content/drive/MyDrive/MotionPrivacyResearch/datasets")

['nturgbd_skeletons_s001_to_s017']

In [6]:
import os

dataset_path = "/content/drive/MyDrive/MotionPrivacyResearch/datasets/nturgbd_skeletons_s001_to_s017"

files = os.listdir(dataset_path)

print(files)

['nturgb+d_skeletons']


In [7]:
import os

dataset_path = "/content/drive/MyDrive/MotionPrivacyResearch/datasets/nturgbd_skeletons_s001_to_s017"

files = os.listdir(dataset_path)

print("Total files:", len(files))
print(files[:10])

Total files: 1
['nturgb+d_skeletons']


In [9]:
import os

dataset_path = "/content/drive/MyDrive/MotionPrivacyResearch/datasets/nturgbd_skeletons_s001_to_s017/nturgb+d_skeletons"

files = os.listdir(dataset_path)

print("Total files:", len(files))
print(files[:5])

Total files: 56880
['S015C001P017R001A031.skeleton', 'S007C003P018R002A051.skeleton', 'S014C003P008R001A047.skeleton', 'S012C001P008R001A015.skeleton', 'S003C001P002R001A032.skeleton']


In [10]:
import re

def extract_person_id(filename):
    match = re.search(r'P(\d+)', filename)
    return int(match.group(1))

# test
print(files[0])
print(extract_person_id(files[0]))

S015C001P017R001A031.skeleton
17


In [11]:
import numpy as np

def load_skeleton_file(file_path, max_frames=50):

    with open(file_path, 'r') as f:
        lines = f.readlines()

    num_frames = int(lines[0])
    idx = 1

    skeleton_sequence = []
    frame_count = 0

    for _ in range(num_frames):

        if frame_count >= max_frames:
            break

        num_bodies = int(lines[idx])
        idx += 1

        for _ in range(num_bodies):

            idx += 1  # skip body metadata

            num_joints = int(lines[idx])
            idx += 1

            joints = []

            for _ in range(num_joints):

                joint_info = list(map(float, lines[idx].split()))
                joints.extend(joint_info[:3])
                idx += 1

            skeleton_sequence.append(joints)

        frame_count += 1

    return np.array(skeleton_sequence)

In [12]:
sample_file = os.path.join(dataset_path, files[0])

motion = load_skeleton_file(sample_file)

print("Shape:", motion.shape)

Shape: (42, 75)


In [13]:
X = []
y = []

for file in files[:2000]:

    file_path = os.path.join(dataset_path, file)

    try:
        motion = load_skeleton_file(file_path)

        if len(motion) == 0:
            continue

        # feature extraction
        motion_mean = motion.mean(axis=0)
        motion_std = motion.std(axis=0)

        feature = np.concatenate([motion_mean, motion_std])

        X.append(feature)
        y.append(extract_person_id(file))

    except:
        pass


X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)
print("Labels shape:", y.shape)

Dataset shape: (1990, 150)
Labels shape: (1990,)


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100)

model.fit(X_train, y_train)

preds = model.predict(X_test)

acc = accuracy_score(y_test, preds)

print("Identity Attack Accuracy:", acc)

Identity Attack Accuracy: 0.27638190954773867
